# Scenario Sizing Loop

Runs **all sizing combinations** for **one charging scenario** and saves results to a flat CSV  
that can be used directly as inputs to `financial_model.build_cash_flows()`.


In [ ]:
from pathlib import Path
import sys
import time
import warnings
from itertools import product as iproduct

import numpy as np
import pandas as pd

ROOT = Path('c:/00_Thesis_Code').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings('default')

from config import HYDRO, SOLVER
from data_loader import load_pv_profile, load_hydro_profile, load_market_prices
from revised_bess_model import make_bess
from revised_dispatch_model_VERSAO2 import run_annual_dispatch

print(f'Working from : {ROOT}')
print(f'Solver       : {SOLVER.solver_name}')
print('Imports OK')

## Settings
Change `CHARGING_SCENARIO` and the sizing grids below, then **Run All**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SCENARIO SELECTOR  (one run = one scenario)
# ─────────────────────────────────────────────────────────────────────────────
CHARGING_SCENARIO = 'C'   # 'A' | 'B' | 'C' | 'D'

#Operation Scenario 1:
    # A — PV-only charging,          hydro fixed to grid (FiT active), no grid buy
    # C — PV + hydro charging,       hydro merchant,                   no grid buy

#Operation Scenario 2:
    # B — PV + grid charging,        hydro fixed to grid (FiT active), grid buy OK    
    # D — PV + hydro + grid charging, hydro merchant,                  grid buy OK

# ─────────────────────────────────────────────────────────────────────────────
# SIZING GRID
# ─────────────────────────────────────────────────────────────────────────────
TEST_PV_SIZES       = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0,12.0]   # MW AC (for A and C, remove PV=0.0)
TEST_BESS_POWERS    = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0] # MW
TEST_BESS_DURATIONS = [2.0, 4.0]                            # hours
TEST_C_VALUES       = [0.0, 0.25, 0.50, 0.75, 1.0]         # contract share c

# Operating years 
OPERATING_YEARS = list(range(2027, 2035))   # 2027-2034 (8 years) -> running scenarios A and B
# OPERATING_YEARS = list(range(2035, 2045))   # 2035–2044 (10 years) -> running scenarios C and D

# ─────────────────────────────────────────────────────────────────────────────
# BESS technical parameters
# (mirrored from revised_bess_model.BESSParameters defaults)
# ─────────────────────────────────────────────────────────────────────────────
INITIAL_SOC           = 0.50   # fraction
ROUND_TRIP_EFFICIENCY = 0.88   # one-way η = sqrt(0.88) ≈ 0.938
MAX_CYCLES_PER_DAY    = 1.5    # operational ceiling (≈ 548 / 365)
MAX_CYCLES_PER_YEAR   = 548.0
LIFETIME_YEARS        = 18.0
EOL_CAPACITY_PCT      = 0.65   # end-of-life capacity retention
RAMP_RATE_PCT_PER_MIN = 0.40   # 40% of rated power per minute

# ─────────────────────────────────────────────────────────────────────────────
# DISPATCH / SOLVER settings
# ─────────────────────────────────────────────────────────────────────────────
WINDOW_STEPS        = 7 * 96   # 672 steps — one rolling week at 15-min resolution
MAX_TOTAL_RUNTIME_S = 120      # seconds allocated per annual run
TARGET_MIP_GAP      = 0.03    # 3% optimality gap

GRID_LIMIT_MW = HYDRO.grid_injection_capacity_mw   # 10 MW (from config)

# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT PATHS
# ─────────────────────────────────────────────────────────────────────────────
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV  = RESULTS_DIR / f'dispatch_scenario_{CHARGING_SCENARIO}.csv'
SUMMARY_CSV = RESULTS_DIR / f'dispatch_summary_scenario_{CHARGING_SCENARIO}.csv'

# ─────────────────────────────────────────────────────────────────────────────
# Preview
# ─────────────────────────────────────────────────────────────────────────────

n_combos = len(TEST_PV_SIZES) * len(TEST_C_VALUES) *  sum(
    len(TEST_BESS_DURATIONS) if bess > 0.0 else 1
    for bess in TEST_BESS_POWERS
)

n_runs = n_combos * len(OPERATING_YEARS)

print(f'Charging scenario  : {CHARGING_SCENARIO}')
print(f'PV sizes (MW)      : {TEST_PV_SIZES}')
print(f'BESS powers (MW)   : {TEST_BESS_POWERS}')
print(f'BESS durations (h) : {TEST_BESS_DURATIONS}')
print(f'Contract shares    : {TEST_C_VALUES}')
print(f'Operating years    : {OPERATING_YEARS[0]}–{OPERATING_YEARS[-1]}')
print(f'Sizing combos      : {n_combos}')
print(f'Total dispatch runs: {n_runs:,}')
print(f'Output             : {OUTPUT_CSV}')

In [ ]:
def _make_bess(bess_mw: float, dur_h: float):
    """
    Return a fresh BESSModel with thesis-standard parameters.

    A new instance is created for every (combo, year) so that degradation
    and cycle counting always start from zero — consistent with how the
    financial model treats each operating year independently.
    """
    return make_bess(
        power_mw              = bess_mw,
        duration_h            = dur_h,
        initial_soc           = INITIAL_SOC,
        round_trip_efficiency = ROUND_TRIP_EFFICIENCY,
        max_cycles_per_day    = MAX_CYCLES_PER_DAY,
        max_cycles_per_year   = MAX_CYCLES_PER_YEAR,
        lifetime_years        = LIFETIME_YEARS,
        eol_capacity_pct      = EOL_CAPACITY_PCT,
        ramp_rate_pct_per_min = RAMP_RATE_PCT_PER_MIN,
    )

## Sizing Loop

Results are written to `OUTPUT_CSV` row-by-row (checkpoint after every solve).  
If the kernel is restarted mid-run, re-executing this cell will **resume** from where it stopped.

In [ ]:
# ── CSV columns ──────────────────────────────────────────────────────────────
COLUMNS = [
    # identification
    'charging_scenario', 'pv_mw', 'bess_power_mw', 'bess_duration_h',
    'bess_energy_mwh', 'contract_share', 'year',
    # ── financial model inputs (one value per operating year) ────────────────
    # Pass df_combo[col].tolist() (sorted by year) directly to build_cash_flows()
    'merchant_revenue_eur',   # → annual_merchant_revenues
    'ppa_delivered_mwh',      # → annual_ppa_energy_mwh
    'total_output_mwh',       # → annual_total_output_mwh
    # ── revenue breakdown (EUR / year) ───────────────────────────────────────
    'rev_da_eur',
    'rev_idc_eur',
    'rev_afrr_up_capacity_eur',
    'rev_afrr_up_activation_eur',
    'rev_afrr_dn_capacity_eur',
    'rev_afrr_dn_activation_eur',
    # ── physical dispatch KPIs (MWh / year) ──────────────────────────────────
    'hydro_annual_mwh',
    'pv_annual_mwh',
    'pv_curtailed_mwh',
    'bess_charge_mwh',
    'bess_discharge_mwh',
    'hydro_to_bess_mwh',
    'ppa_shortfall_mwh',
    # ── BESS end-of-year state ────────────────────────────────────────────────
    'total_cycles',
    'final_soc_mwh',
    'final_degradation',
    'total_throughput_mwh',
    # ── run meta ─────────────────────────────────────────────────────────────
    'solve_time_s',
    'status',
]

# ── checkpoint: skip rows already in the CSV ──────────────────────────────
if OUTPUT_CSV.exists() and OUTPUT_CSV.stat().st_size > 0:
    df_done   = pd.read_csv(OUTPUT_CSV)
    done_keys = set(
        zip(df_done['pv_mw'], df_done['bess_power_mw'],
            df_done['bess_duration_h'], df_done['contract_share'],
            df_done['year'])
    )
    write_header = False
    print(f'Resuming — {len(df_done):,} rows already saved to {OUTPUT_CSV.name}')
else:
    done_keys    = set()
    write_header = True
    print('Starting fresh run.')

# ── open file for streaming writes ────────────────────────────────────────
csv_fh = open(OUTPUT_CSV, 'a', newline='', encoding='utf-8')
if write_header:
    csv_fh.write(','.join(COLUMNS) + '\n')

# ── main loop ─────────────────────────────────────────────────────────────
t_global  = time.time()
combo_num = 0
combos = [
    (pv_mw, bess_mw, dur_h, c)
    for pv_mw, bess_mw, dur_h, c in iproduct(
        TEST_PV_SIZES, TEST_BESS_POWERS, TEST_BESS_DURATIONS, TEST_C_VALUES)
    if bess_mw > 0.0 or dur_h == TEST_BESS_DURATIONS[0]
]
n_combos = len(combos)

try:
    for pv_mw, bess_mw, dur_h, c in combos:

        combo_num += 1
        bess_mwh  = bess_mw * dur_h
        elapsed   = (time.time() - t_global) / 60
        print(f'\n[{combo_num}/{n_combos}]  '
              f'PV={pv_mw} MW | BESS={bess_mw} MW × {dur_h} h | '
              f'c={c:.0%}  (elapsed {elapsed:.1f} min)')

        for year in OPERATING_YEARS:
            key = (pv_mw, bess_mw, dur_h, c, year)
            if key in done_keys:
                print(f'  {year} — skipped (already done)')
                continue

            t0 = time.time()
            try:
                pv_profile    = load_pv_profile(pv_mw, year)
                hydro_profile = load_hydro_profile(year)
                prices        = load_market_prices(year)
                bess          = _make_bess(bess_mw, dur_h)

                res = run_annual_dispatch(
                    prices              = prices,
                    pv_profile          = pv_profile,
                    hydro_profile       = hydro_profile,
                    bess_model          = bess,
                    pv_installed_mw     = pv_mw,
                    grid_limit_mw       = GRID_LIMIT_MW,
                    contract_share      = c,
                    charging_scenario   = CHARGING_SCENARIO,
                    window_steps        = WINDOW_STEPS,
                    max_total_runtime_s = MAX_TOTAL_RUNTIME_S,
                    target_mip_gap      = TARGET_MIP_GAP,
                )

                rev = res['annual_revenues']
                merchant_rev = sum(rev.values())
                dt = round(time.time() - t0, 1)

                row_vals = [
                    CHARGING_SCENARIO, pv_mw, bess_mw, dur_h, bess_mwh, c, year,
                    # financial model inputs
                    merchant_rev,
                    res['ppa_delivered_mwh'],
                    res['total_output_mwh'],
                    # revenue breakdown
                    rev['da'],
                    rev['idc'],
                    rev['afrr_up_capacity'],
                    rev['afrr_up_activation'],
                    rev['afrr_down_capacity'],
                    rev['afrr_down_activation'],
                    # physical KPIs
                    res['hydro_annual_mwh'],
                    res['pv_annual_mwh'],
                    res['pv_curtailed_mwh'],
                    res['bess_charge_mwh'],
                    res['bess_discharge_mwh'],
                    res['hydro_to_bess_mwh'],
                    res['ppa_shortfall_mwh'],
                    # BESS state
                    res['total_cycles'],
                    res['final_soc_mwh'],
                    res['final_degradation'],
                    res['total_throughput_mwh'],
                    #afrr factors
                    res['annual_acceptance_factor'],
                    res['annual_afrr_up_factor'],
                    res['annual_afrr_down_factor'],
                    # meta
                    dt,
                    'ok',
                ]
                print(f'  {year}  ok  ({dt}s)')

            except Exception as exc:
                dt = round(time.time() - t0, 1)
                err_msg = str(exc).replace(',', ';').replace('\n', ' ')
                print(f'  {year}  ERROR: {err_msg}')
                row_vals = (
                    [CHARGING_SCENARIO, pv_mw, bess_mw, dur_h, bess_mwh, c, year]
                    + [''] * (len(COLUMNS) - 9)
                    + [dt, f'ERROR: {err_msg}']
                )

            # flush immediately — checkpoint
            csv_fh.write(','.join(str(v) for v in row_vals) + '\n')
            csv_fh.flush()
            done_keys.add(key)

finally:
    csv_fh.close()

total_min = (time.time() - t_global) / 60
print(f'\n{"─"*60}')
print(f'Loop complete in {total_min:.1f} min.')
print(f'Output: {OUTPUT_CSV}')

## Inspect Results

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
df_ok = df[df['status'] == 'ok'].copy()

print(f'Total rows   : {len(df):,}')
print(f'Successful   : {len(df_ok):,}')
print(f'Errors       : {len(df) - len(df_ok):,}')

if (len(df) - len(df_ok)) > 0:
    print('\nFailed rows:')
    display(df[df['status'] != 'ok'][['pv_mw','bess_power_mw','bess_duration_h',
                                       'contract_share','year','status']])

print()
numeric_cols = ['merchant_revenue_eur', 'ppa_delivered_mwh', 'total_output_mwh',
                'pv_annual_mwh', 'hydro_annual_mwh', 'bess_discharge_mwh',
                'ppa_shortfall_mwh', 'total_cycles', 'final_degradation']
df_ok[numeric_cols] = df_ok[numeric_cols].apply(pd.to_numeric, errors='coerce')
display(df_ok[numeric_cols].describe().T[['count','mean','min','max']].round(2))